# Summary of HPC runs
Pairs each LSF job file `job_out/<jobid>.out` with its run folder `runs/<jobid>_<case>/result.txt` (matched on the job id) and collects everything into one DataFrame.

- **From `job_out/<jobid>.out`:** job name, case, cores, soft limit, LSF status (Done / Exited / killed by limit), host, queue, wall time, CPU time, max memory.
- **From `runs/<jobid>_<case>/result.txt`:** iterations, last `Val`, PoA bound, CCE bound, number of joint actions, whether the time limit was reached, and the last line of a traceback if the run crashed.

Run this notebook from the repo folder (the one that contains `job_out/` and `runs/`).

In [ ]:
import re
from pathlib import Path
from datetime import datetime
import pandas as pd

ROOT = Path.cwd()                 # repo folder; change if the notebook lives elsewhere
JOB_OUT = ROOT / "job_out"
RUNS = ROOT / "runs"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
print("job_out:", JOB_OUT.exists(), "| runs:", RUNS.exists())

## Parsers

In [ ]:
LSF_TIME = "%a %b %d %H:%M:%S %Y"

def read(path):
    return Path(path).read_text(errors="replace")

def num(pattern, text, cast=float):
    m = re.search(pattern, text, flags=re.M)
    return cast(m.group(1)) if m else None

def parse_time(pattern, text):
    m = re.search(pattern, text)
    if not m:
        return None
    try:
        return datetime.strptime(" ".join(m.group(1).split()), LSF_TIME)
    except ValueError:
        return None


def parse_job_out(path):
    """LSF job file: our own header lines + LSF's end-of-job report."""
    t = read(path)
    d = {}

    # --- header printed by run_case.sh ---
    m = re.search(r"Job (\d+) on (\S+), cores=(\d+), case='(.*?)', soft limit=(\d+(?:\.\d+)?)s", t)
    if m:
        d.update(host=m.group(2), cores=int(m.group(3)), case_file=m.group(4), soft_limit_s=float(m.group(5)))
    d["script_exit"] = num(r"main\.py exited with status (\d+)", t, int)

    # --- LSF report ---
    m = re.search(r"Subject: Job \d+: <(.*?)> in cluster <.*?> (\S+)", t)
    d["job_name"], d["lsf_state"] = (m.group(1), m.group(2)) if m else (None, "RUNNING/unfinished")
    d["queue"] = (re.search(r"in queue <(.*?)>", t) or [None, None])[1]
    started = parse_time(r"Started at (.+)", t)
    ended = parse_time(r"Terminated at (.+)", t)
    submitted = parse_time(r"submitted from host <.*?> by user <.*?> in cluster <.*?> at (.+)", t)
    d["wall_s"] = (ended - started).total_seconds() if started and ended else None
    d["queue_wait_s"] = (started - submitted).total_seconds() if started and submitted else None
    d["started"] = started

    # how did it end?
    if "TERM_RUNLIMIT" in t:
        d["end_reason"] = "hard time limit (TERM_RUNLIMIT)"
    elif "TERM_MEMLIMIT" in t:
        d["end_reason"] = "memory limit (TERM_MEMLIMIT)"
    elif "TERM_" in t:
        d["end_reason"] = re.search(r"(TERM_\w+)", t).group(1)
    elif "Successfully completed" in t:
        d["end_reason"] = "completed"
    elif re.search(r"Exited with exit code (\d+)", t):
        d["end_reason"] = "exit code " + re.search(r"Exited with exit code (\d+)", t).group(1)
    else:
        d["end_reason"] = None

    d["cpu_s"] = num(r"CPU time :\s+([\d.]+) sec", t)
    d["max_mem_MB"] = num(r"Max Memory :\s+([\d.]+) MB", t)       # '-' on very short jobs -> None
    d["run_time_s"] = num(r"Run time :\s+([\d.]+) sec", t)
    d["req_mem_MB"] = num(r"Total Requested Memory :\s+([\d.]+) MB", t)
    return d


def parse_result(path):
    """main.py printout."""
    t = read(path)
    d = {}
    d["iterations"] = num(r"Iterations:\s+(\d+)", t, int)
    if d["iterations"] is None:                      # unfinished run: count the iteration lines so far
        d["iterations"] = len(re.findall(r"^Iteration:\s+\d+", t, flags=re.M)) or None
    vals = re.findall(r"^Val:\s+(\S+)", t, flags=re.M)
    d["last_val"] = float(vals[-1]) if vals else None
    d["poa_bound"] = num(r"^poa_bound .*?:\s+(\S+)", t)
    d["cce_bound"] = num(r"^CCE poa_bound .*?:\s+(\S+)", t)
    d["joint_actions"] = num(r"Number of joint actions .*?:\s+(\d+)", t, int)
    d["converged"] = d["poa_bound"] is not None
    d["time_limit_hit"] = "Time limit reached" in t
    err = None
    if "Traceback" in t:
        lines = [l for l in t.strip().splitlines() if l.strip()]
        err = lines[-1].strip()
    d["error"] = err
    d["result_lines"] = len(t.splitlines())
    return d

## Pair the files on job id

In [ ]:
# job files: only "<jobid>.out" (skips test files such as lic_<jobid>.out)
job_rows = {}
for p in sorted(JOB_OUT.glob("*.out")) if JOB_OUT.exists() else []:
    if re.fullmatch(r"\d+", p.stem):
        job_rows[int(p.stem)] = parse_job_out(p)

# run folders: "<jobid>_<case name>"
run_rows = {}
for p in sorted(RUNS.iterdir()) if RUNS.exists() else []:
    m = re.fullmatch(r"(\d+)_(.+)", p.name)
    if m and p.is_dir():
        row = {"run_dir": p.name, "case_name": m.group(2)}
        res = p / "result.txt"
        row.update(parse_result(res) if res.exists() else {"error": "no result.txt"})
        run_rows[int(m.group(1))] = row

jobs = pd.DataFrame.from_dict(job_rows, orient="index")
runs = pd.DataFrame.from_dict(run_rows, orient="index")
df = jobs.join(runs, how="outer", rsuffix="_run")
df.index.name = "jobid"
df = df.reset_index()

# unpaired entries (job file without run folder, or the reverse)
df["paired"] = df["run_dir"].notna() & df["lsf_state"].notna()

# case size for sorting: ..._case118_ieee -> 118
src = df["case_name"].fillna(df["case_file"].fillna("").str.replace(r".*/", "", regex=True).str.replace(r"\.m$", "", regex=True))
df["case"] = src.str.extract(r"case(\d+)", expand=False).astype("Int64")
df["case_name"] = src
print(f"{len(df)} jobs  |  paired: {df['paired'].sum()}  |  unpaired: {(~df['paired']).sum()}")

## Summary table

In [ ]:
cols = ["jobid", "case", "end_reason", "converged", "iterations", "last_val", "poa_bound", "cce_bound",
        "joint_actions", "wall_s", "cpu_s", "max_mem_MB", "cores", "soft_limit_s", "queue_wait_s", "time_limit_hit", "paired"]
cols = [c for c in cols if c in df.columns]
summary = df.sort_values(["case", "jobid"])[cols].reset_index(drop=True)
summary

## Failures and unpaired files

In [ ]:
problems = df[(df["error"].notna()) | (~df["paired"]) | (df["end_reason"].fillna("") != "completed")]
problems.sort_values(["case", "jobid"])[["jobid", "case", "lsf_state", "end_reason", "script_exit", "error", "run_dir"]] \
    if len(problems) else print("No problems found.")

## Latest run per case
If you submitted a case more than once, this keeps the most recent job id.

In [ ]:
latest = (df[df["paired"]].sort_values("jobid").groupby("case", as_index=False).tail(1)
          .sort_values("case")[["case", "jobid", "end_reason", "converged", "iterations", "poa_bound", "cce_bound", "wall_s", "max_mem_MB"]])
latest.reset_index(drop=True)

## Save

In [ ]:
summary.to_csv(ROOT / "runs_summary.csv", index=False)
print("saved", ROOT / "runs_summary.csv")